# HeritAgent — EXP-0001 grafikleri (Colab, CPU çalışma zamanı)

Bir EXP-0001 koşusunun makale grafiklerini, Drive'daki aynasından yeniden üretir. Koşu notebook'u aynı grafikleri her oturumun sonunda zaten çizer; bu notebook onları sonradan, GPU'suz yeniden çizmek içindir (örneğin bitmiş bir koşu için ya da grafik kodu değiştiyse).

**Ne yapar:** koşunun aynasındaki en son tam kopyayı **yalnızca okur** (her dosya hash'iyle denetlenir); aynaya ve koşuya hiçbir şey yazmaz. Grafikleri `heritagent/runs/<RUN_ID>/figures/` altına yazar:
- her grafik için PDF (makale için), PNG ve CSV (grafiğin tablo görünümü);
- `captions.md`: açıklamalar ve raporun taşıması gereken notlar;
- `figures.json`: okunan ve yazılan her dosyanın sha256'sı ve grafikleri çizen kodun commit'i.

Bu notebook, deney notebook'larının kilitli ortamını kurmaz: grafikleri Colab'ın kendi Python'u ve matplotlib'iyle çizer. Bu yüzden PNG ve PDF dosyaları, koşu notebook'unun kilitli ortamdaki matplotlib ile çizdiklerinden bayt bayt farklı olabilir; CSV'lerdeki sayılar aynıdır.

Grafikler yalnızca koşunun kaydettiği sayıları gösterir. Aralık yalnızca dört havuzlanmış kontrastta vardır (ADR-0006). Bitmemiş ya da eksik bir koşunun grafikleri betimseldir ve üstlerinde bu yazar. Hiçbir sonuç "doğrulandı" diye etiketlenmez (SPEC §32).

**Ne zaman:** o koşu başka bir sekmede ya da oturumda **çalışmıyorken**. Koşu notebook'u da her oturumun sonunda aynı klasöre yazar; ikisi aynı anda yazmasın. Çalışan bir koşunun aynası okunursa grafikler son tam kopyayı gösterir ve UNFINISHED diye işaretlenir; nadiren, okuma sırasında silinen bir kopya parçası yüzünden durabilir, o zaman koşu bittikten sonra yeniden çalıştırın.

**Nasıl çalıştırılır:**
1. Çalışma zamanı > Çalışma zamanı türünü değiştir > **CPU**.
2. Aşağıdaki `RUN_ID`'yi grafiklerini istediğiniz koşununkiyle aynı yapın.
3. Çalışma zamanı > **Tümünü çalıştır**. Drive izni isteyince onaylayın.
4. Bir hücre hata verirse hata metnini bana gönderin.

In [ ]:
# Ayarlar
RUN_ID = "EXP-0001-run-1"  # grafiklerini istediğiniz koşu (koşu notebook'undaki RUN_ID)
REPO_URL = "https://github.com/sametyilmaztemel/heritagent.git"
CODE_REF = "main"          # grafik kodunun commit'i; koşuyu yalnızca okur, figures.json'a yazılır
DRIVE_ROOT = "/content/drive/MyDrive/heritagent"
AUTO_RELEASE = True        # bitince ya da bir hata olursa çalışma zamanını bırak

In [ ]:
# 0) Bir hücre hata verirse "Tümünü çalıştır" orada durur. AUTO_RELEASE açıksa sunucu günlüğü
# Drive'a kopyalanır, Drive boşaltılır ve çalışma zamanı bırakılır: hatadan sonra boşta
# bekleyen çalışma zamanı da CU harcar. Bir deney komutu hâlâ çalışıyorsa ya da son kopyası
# Drive'a alınamamış bir koşu varsa çalışma zamanı BIRAKILMAZ (o iş yalnızca bu makinededir).
import glob, os, shutil, time
from google.colab import drive, runtime as colab_runtime  # kurulumdan önce yüklenir


def unmirrored():
    """Son kopyası Drive'a alınamamış yerel koşu klasörleri (ADR-0008 D5)."""
    return sorted(os.path.dirname(p) for p in glob.glob("/content/*/UNMIRRORED.json"))


def running_commands():
    """Bu makinede hâlâ çalışan run/pilot/snapshot komutlarının süreç numaraları. Çekirdek
    yeniden başladıysa önceki komut kendiliğinden durur, ama süren bölümlerini kaydedip son
    kopyayı alana dek (birkaç dakika) çalışır."""
    found = []
    for entry in os.listdir("/proc"):
        if not entry.isdigit() or int(entry) == os.getpid():
            continue
        try:
            with open(f"/proc/{entry}/cmdline", "rb") as handle:
                argv = handle.read().split(b"\0")
        except OSError:
            continue  # bu arada bitti
        if b"evaluation.exp0001.cli" in argv:
            command = argv[argv.index(b"evaluation.exp0001.cli") + 1:][:1]
            if command and command[0] in (b"run", b"pilot", b"snapshot"):
                found.append(int(entry))
    return sorted(found)


def release():
    """Drive'ı boşaltır ve çalışma zamanını bırakır; bir komut hâlâ çalışıyorsa ya da aynalanmamış bir
    koşu varsa bırakmaz."""
    busy = running_commands()
    if busy:
        print("ÇALIŞMA ZAMANI BIRAKILMADI: bir deney komutu hâlâ çalışıyor (süreç", busy, "). Çekirdek "
              "yeniden başladıysa bu önceki komuttur: süren bölümlerini kaydedip son kopyayı "
              "alıyor.")
        print("Birkaç dakika bekleyin, sonra o hücreyi yeniden çalıştırın (kaldığı yerden "
              "sürer) ya da release() çalıştırın.")
        print("15 dakikada bitmezse: import signal; os.killpg(<süreç>, signal.SIGTERM) ile "
              "sonlandırın. Son kopyadan sonraki kayıtlar bu makinede kalır (UNMIRRORED.json); "
              "ardından cli('snapshot', '--run-dir', <klasör>, '--mirror-dir', <ayna>) "
              "çalıştırın (ayna: koşu için RUN_DRIVE + '/mirror', pilot için PILOT_DRIVE + "
              "'/mirror'). Runtime > Disconnect and delete runtime ise o kayıtları siler.")
        return
    kept = unmirrored()
    if kept:
        print("ÇALIŞMA ZAMANI BIRAKILMADI: son kopyası Drive'a alınamamış iş var:", kept)
        print("1) Drive'ı yeniden bağlayın: drive.mount('/content/drive', force_remount=True)")
        print("2) Her klasör için: cli('snapshot', '--run-dir', <klasör>, '--mirror-dir', <ayna>)")
        print("   (ayna: koşu için RUN_DRIVE + '/mirror', pilot için PILOT_DRIVE + '/mirror')")
        print("3) Kalan hücreleri (durum, rapor kopyası) çalıştırın; sonra release() çalıştırın: "
              "Drive'ı boşaltır ve çalışma zamanını bırakır.")
        return
    try:  # Drive'a yazılanlar makine kapanmadan yüklensin; en fazla 10 dakika beklenir
        drive.flush_and_unmount(timeout_ms=600_000)
    except BaseException as exc:  # bağlanmamış, takılmış ya da durdurulmuş: çalışma zamanı yine bırakılır
        print("Drive boşaltılamadı:", repr(exc))
    print("Çalışma zamanı bırakılıyor.")
    try:
        colab_runtime.unassign()
    except Exception as exc:
        print("Çalışma zamanı bırakılamadı; bağlantıyı kendiniz kesin:", exc)


def _on_error(shell, etype, value, tb, tb_offset=None):
    try:
        shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
        if not AUTO_RELEASE:
            print("AUTO_RELEASE = False: çalışma zamanı bağlı kalıyor; işiniz bitince bağlantıyı kesin.")
            return
        try:
            if os.path.isdir(DRIVE_ROOT) and os.path.exists("/content/vllm-server.log"):
                saved = f"{DRIVE_ROOT}/logs/vllm-server-{time.strftime('%Y%m%d-%H%M%S')}.log"
                os.makedirs(os.path.dirname(saved), exist_ok=True)
                shutil.copy("/content/vllm-server.log", saved)
                print("Sunucu günlüğü Drive'da:", saved)
        except Exception as exc:
            print("Sunucu günlüğü Drive'a kopyalanamadı:", exc)
        if etype.__name__ != "Bitti":
            print("Bir adım başarısız oldu. Hata metnini bana gönderin.")
    finally:  # günlükteki ya da Drive'daki bir hata çalışma zamanının bırakılmasını engellemez
        if AUTO_RELEASE:
            release()


get_ipython().set_custom_exc((Exception,), _on_error)

In [ ]:
# 2) Google Drive: kalıcı kayıtlar burada durur (oturum kapanınca yerel disk silinir).
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)


class Bitti(Exception):
    pass

In [ ]:
# 2b) Koşunun aynası: koşu notebook'unun yazdığı yer.
RUN_DRIVE = f"{DRIVE_ROOT}/runs/{RUN_ID}"
if not os.path.isdir(f"{RUN_DRIVE}/mirror"):
    raise RuntimeError(f"{RUN_DRIVE}/mirror yok: RUN_ID doğru mu?")

In [ ]:
# 3) Kod: depoyu klonla ve CODE_REF noktasına geç.
import os, subprocess
REPO = "/content/heritagent"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--quiet", "origin"], check=True)


def checkout(ref):
    target = f"origin/{ref}" if subprocess.run(
        ["git", "-C", REPO, "rev-parse", "--verify", "--quiet", f"origin/{ref}"],
        capture_output=True).returncode == 0 else ref
    subprocess.run(["git", "-C", REPO, "checkout", "--quiet", "--detach", target], check=True)
    return subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True,
                          text=True, check=True).stdout.strip()


COMMIT = checkout(CODE_REF)
os.chdir(REPO)
print("Kod commit'i:", COMMIT)

In [ ]:
# 4) Kurulum (~1-2 dk): yalnızca depo ve matplotlib; model, vLLM ya da ALFWorld kurulmaz.
import os, signal, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", f"{REPO}[figures]"],
               check=True)
PY = sys.executable  # grafikler Colab'ın kendi Python'u ve matplotlib'iyle çizilir: kilitli ortam


def command_environment():  # gerekmez (çizim, kaydı yazılmış bir koşuyu yalnızca okur)
    return dict(os.environ)


def cli(*args):
    """Deneyin bir komutunu çalıştırır ve çıktısını canlı yazar. Komut başarısız olursa hata
    verir; böylece "Tümünü çalıştır" sonraki (para harcayan) hücrelere geçmez.
    Durdur düğmesi: komut, süre sınırındaki gibi durur (sürmekte olan bölümler kaydedilir,
    son kopya Drive'a alınır); ikinci basış onu hemen sonlandırır."""
    command = [PY, "-u", "-m", "evaluation.exp0001.cli", *map(str, args)]
    # kendi süreç grubunda: durdurma sinyali ona yalnızca buradan, bir kez ulaşır; çekirdek
    # yeniden başlarsa ya da ölürse komut da aynı şekilde durur (HERITAGENT_STOP_WITH_PARENT)
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, start_new_session=True,
                          env={**command_environment(),
                               "HERITAGENT_STOP_WITH_PARENT": str(os.getpid())}) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
        except KeyboardInterrupt:
            process.send_signal(signal.SIGINT)
            print("Durduruluyor: süren bölümler kaydediliyor ve son kopya alınıyor (birkaç "
                  "dakika sürebilir). Hemen sonlandırmak için yeniden basın.", flush=True)
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
            except KeyboardInterrupt:
                os.killpg(process.pid, signal.SIGTERM)
                print("Sonlandırıldı. Bu bir run, pilot ya da snapshot komutuysa son kopyadan "
                      "sonraki kayıtlar yalnızca bu makinede olabilir ve UNMIRRORED.json "
                      "bırakılmıştır (unmirrored() ile bakın): çalışma zamanı, cli('snapshot', "
                      "'--run-dir', <klasör>, '--mirror-dir', <ayna>) çalışana dek "
                      "bırakılmaz.", flush=True)
            raise
    if process.returncode == 3:  # cli.EXIT_UNMIRRORED
        raise RuntimeError(f"`cli {args[0]}`: son kopya Drive'a alınamadı (UNMIRRORED); en yeni "
                           f"kayıtlar yalnızca bu makinede. Çalışma zamanı bırakılmayacak.")
    if process.returncode != 0:
        raise RuntimeError(f"`cli {args[0]}` başarısız oldu (çıkış kodu {process.returncode}); "
                           f"sonraki hücreler çalıştırılmadı.")

In [ ]:
# 5) Grafikler: aynanın en son tam kopyasından (yalnızca okur).
import shutil
from IPython.display import Image, display
FIGURES = "/content/exp0001-figures"
cli("figures", "--mirror-dir", f"{RUN_DRIVE}/mirror", "--out-dir", FIGURES)
shutil.rmtree(f"{RUN_DRIVE}/figures", ignore_errors=True)
shutil.copytree(FIGURES, f"{RUN_DRIVE}/figures")
for name in sorted(os.listdir(FIGURES)):
    if name.endswith(".png"):
        display(Image(filename=f"{FIGURES}/{name}"))
print("Grafikler:", f"{RUN_DRIVE}/figures")

In [ ]:
# Son) Drive'ı boşalt ve çalışma zamanını bırak (CPU çalışma zamanı da, az da olsa, CU harcar).
# İncelemek isterseniz AUTO_RELEASE = False yapın. (Bir hücre hata verirse 0. hücre aynısını yapar.)
if AUTO_RELEASE:
    release()